# Парсер топ-250 фильмов IMDb

Скрипт собирает данные о фильмах из чарта IMDb Top 250:
- Ранг
- Название
- Год выпуска
- Страна производства
- Жанры (первые три)
- Описание сюжета

Результат сохраняется в `imdb_results.csv`.

## Установка зависимостей

```bash
pip install playwright beautifulsoup4 playwright-stealth
playwright install chromium
```

In [83]:
# Ячейка 1: Импорты и настройки
import json
import csv
import time
import random
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

try:
    from playwright_stealth import stealth_async
    HAS_STEALTH = True
except ImportError:
    HAS_STEALTH = False

BASE_URL = "https://www.imdb.com/chart/top/"
CHROMIUM_PATH = "/usr/bin/chromium"   # измените под свой путь или удалите строку
MAX_CONCURRENT = 2
LIMIT = 250

In [84]:
# Ячейка 2: Вспомогательные функции
def find_genres_recursive(obj):
    if isinstance(obj, dict):
        if "genres" in obj and isinstance(obj["genres"], list):
            result = [g.get("text", "") for g in obj["genres"] if g.get("text")]
            if result:
                return result
        for v in obj.values():
            result = find_genres_recursive(v)
            if result:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_genres_recursive(item)
            if result:
                return result
    return []

def extract_genres(soup):
    genres_list = []
    json_ld = soup.find("script", {"type": "application/ld+json"})
    if json_ld:
        try:
            ld_data = json.loads(json_ld.string)
            if "genre" in ld_data:
                raw = ld_data["genre"]
                genres_list = [raw] if isinstance(raw, str) else list(raw)
        except Exception:
            pass
    if not genres_list:
        next_data_tag = soup.find("script", {"id": "__NEXT_DATA__"})
        if next_data_tag:
            try:
                nd = json.loads(next_data_tag.string)
                genres_list = find_genres_recursive(nd)
            except Exception:
                pass
    if not genres_list:
        seen = set()
        for a in soup.find_all("a", href=lambda x: x and "/search/title?genres=" in x):
            t = a.get_text(strip=True)
            if t and t not in seen:
                seen.add(t)
                genres_list.append(t)
    if not genres_list:
        for chip in soup.select('[data-testid="genres"] a, [class*="ipc-chip"] span'):
            t = chip.get_text(strip=True)
            if t and t not in genres_list:
                genres_list.append(t)
    return genres_list[:3]

In [85]:
# Ячейка 3: Асинхронный сбор данных одного фильма
async def scrape_movie_details(context, item_data, semaphore):
    async with semaphore:
        page = await context.new_page()
        if HAS_STEALTH:
            await stealth_async(page)
        url = f"https://www.imdb.com/title/{item_data['id']}/"
        try:
            await asyncio.sleep(random.uniform(0.8, 1.5))
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_selector("h1", timeout=60000)
            html = await page.content()
            s = BeautifulSoup(html, "html.parser")
            desc_tag = (
                s.find("span", {"data-testid": "plot-xs_to_m"})
                or s.find("span", {"data-testid": "plot-xl"})
                or s.select_one('[data-testid="plot"]')
            )
            description = desc_tag.get_text(strip=True) if desc_tag else "No description"
            genres_list = extract_genres(s)
            genres = ", ".join(genres_list) if genres_list else "N/A"
            country = "N/A"
            country_box = s.find("li", {"data-testid": "title-details-origin"})
            if country_box:
                country_link = country_box.find("a")
                if country_link:
                    country = country_link.get_text(strip=True)
            print(f"✅ [{item_data['rank']}] {item_data['title']} | Жанры: {genres} | Страна: {country}")
            await page.close()
            return {
                "rank": item_data["rank"],
                "title": item_data["title"],
                "year": item_data["year"],
                "country": country,
                "genres": genres,
                "description": description,
            }
        except Exception as e:
            print(f"❌ Ошибка на {item_data['title']}: {e}")
            await page.close()
            return None

In [86]:
# Ячейка 4: Асинхронная главная функция
async def main():
    start_time = time.time()
    all_movies_list = []

    async with async_playwright() as p:
        print("🔍 Запуск браузера для получения списка...")
        browser = await p.chromium.launch(
            executable_path=CHROMIUM_PATH,
            headless=False,
            args=["--disable-blink-features=AutomationControlled"],
        )
        context = await browser.new_context()
        page = await context.new_page()
        if HAS_STEALTH:
            await stealth_async(page)

        try:
            await page.goto(BASE_URL, wait_until="networkidle", timeout=120000)
            await page.wait_for_selector("li.ipc-metadata-list-summary-item", timeout=20000)
            script_content = await page.locator("script#__NEXT_DATA__").inner_text()
            data = json.loads(script_content)
            try:
                edges = data["props"]["pageProps"]["pageData"]["chartTitles"]["edges"]
            except KeyError:
                edges = data["props"]["pageProps"]["mainColumnData"]["chartTitles"]["edges"]
            for i, item in enumerate(edges[:LIMIT]):
                node = item["node"]
                all_movies_list.append({
                    "id": node["id"],
                    "rank": i + 1,
                    "title": node["titleText"]["text"],
                    "year": node["releaseYear"]["year"],
                })
            print(f"🎯 Получено {len(all_movies_list)} фильмов. Начинаем сбор деталей...")
        except Exception as e:
            print(f"🚨 Ошибка при получении списка: {e}")
            return
        finally:
            await browser.close()

    if not all_movies_list:
        return

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            executable_path=CHROMIUM_PATH,
            headless=False,
            args=["--disable-blink-features=AutomationControlled"],
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
        )
        semaphore = asyncio.Semaphore(MAX_CONCURRENT)
        tasks = [scrape_movie_details(context, item, semaphore) for item in all_movies_list]
        results = await asyncio.gather(*tasks)
        await browser.close()

    final_results = [r for r in results if r is not None]
    final_results.sort(key=lambda x: x["rank"])

    with open("imdb_results.csv", "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["rank", "title", "year", "country", "genres", "description"])
        writer.writeheader()
        writer.writerows(final_results)

    elapsed = round((time.time() - start_time) / 60, 2)
    print(f"\n✨ Готово! {len(final_results)} фильмов сохранено в imdb_results.csv за {elapsed} мин.")

In [ ]:
# Ячейка 5: Запуск
await main()

🔍 Запуск браузера для получения списка...
🎯 Получено 250 фильмов. Начинаем сбор деталей...
✅ [2] The Godfather | Жанры: Crime, Drama | Страна: United States
✅ [1] The Shawshank Redemption | Жанры: Drama | Страна: United States
✅ [3] The Dark Knight | Жанры: Action, Crime, Drama | Страна: United States
✅ [4] The Godfather Part II | Жанры: Crime, Drama | Страна: United States
✅ [5] 12 Angry Men | Жанры: Crime, Drama | Страна: United States
✅ [6] The Lord of the Rings: The Return of the King | Жанры: Adventure, Drama, Fantasy | Страна: New Zealand


Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed\nCall log:\n  - navigating to "https://www.imdb.com/chart/top/", waiting until "networkidle"\n')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed
Call log:
  - navigating to "https://www.imdb.com/chart/top/", waiting until "networkidle"



✅ [7] Schindler's List | Жанры: Biography, Drama, History | Страна: United States
✅ [8] The Lord of the Rings: The Fellowship of the Ring | Жанры: Adventure, Drama, Fantasy | Страна: New Zealand
✅ [9] Pulp Fiction | Жанры: Crime, Drama | Страна: United States
✅ [10] The Good, the Bad and the Ugly | Жанры: Adventure, Drama, Western | Страна: Italy
✅ [11] The Lord of the Rings: The Two Towers | Жанры: Adventure, Drama, Fantasy | Страна: New Zealand
✅ [12] Forrest Gump | Жанры: Drama, Romance | Страна: United States
✅ [13] Fight Club | Жанры: Crime, Drama, Thriller | Страна: United States
✅ [14] Inception | Жанры: Action, Adventure, Sci-Fi | Страна: United Kingdom
✅ [15] Star Wars: Episode V - The Empire Strikes Back | Жанры: Action, Adventure, Fantasy | Страна: United States
✅ [16] The Matrix | Жанры: Action, Sci-Fi | Страна: United States
✅ [17] GoodFellas | Жанры: Biography, Crime, Drama | Страна: United States
✅ [18] Interstellar | Жанры: Adventure, Drama, Sci-Fi | Страна: United Stat